# OptDebug — which operation is at fault

Provenance says which records produced a wrong result. It says nothing about which
part of the query mishandled them. This scores operations the way spectrum-based
fault localisation scores lines of code. The answer is in the **code**.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("optdebug-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## Rank the operations

Narrowing the failing records first is what makes the guilty operation stand out, so the base table is named.

In [ ]:
QUERY = ("SELECT cid, SUM(CASE WHEN amount > 1000 THEN -amount ELSE amount END) AS total "
    "FROM orders GROUP BY cid")

result = bigasterisk.optdebug(spark).localize(
    QUERY, faulty_where="total < 0", base_table="orders")

print("failing records:", result.minimised_from, "->", result.failing_witnesses)
for op in result.ranked:
    print(op)

## Check

The faulty branch is the `CASE WHEN` arm only the outlier takes.

In [ ]:
top = result.prime
assert top.is_branch and "1000" in top.branch, repr(top)
assert abs(top.score - 1.0) < 1e-9
assert top.passing_witnesses == 0
print("OK")